# 01 - Federated data partition (Paper 4, Fase 1)

**Tujuan:** tiap *klien* menerapkan kamus **SFM (public prior dari Paper 1)** ke
datanya SENDIRI -> 9 fitur SFM lokal. Client-CIC & Client-UNSW TIDAK bertukar data;
yang disepakati hanya *skema* (9 fitur + definisi). Lihat `documentation_federated.md`
Bagian A.1 (SFM = public prior, bukan hasil belajar data gabungan).

**Output:** `fed_out/dataset_stats_fed.csv` (statistik per-klien, ANGKA NYATA) +
partisi SFM per-klien ke `fed_out/` dan S3 `unsw-far/federated/data/{cic,unsw}/`.

> Jalankan di SageMaker (butuh CIC `cleaned_100.pkl` + UNSW CSV). JANGAN mengarang angka.

## 0. Setup & kamus SFM (public prior, warisi Paper 1)

In [ ]:
import importlib, sys, subprocess
for m in ('pandas','numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, pickle, numpy as np, pandas as pd
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEED=42

# --- Kamus SFM (public prior; identik Paper 1/nb17). (CIC_col, UNSW_col) per fitur ---
SFM = {
  'duration' : ('Flow Duration',   'dur'),
  'fwd_pkts' : ('Tot Fwd Pkts',    'spkts'),
  'bwd_pkts' : ('Tot Bwd Pkts',    'dpkts'),
  'fwd_bytes': ('TotLen Fwd Pkts', 'sbytes'),
  'bwd_bytes': ('TotLen Bwd Pkts', 'dbytes'),
  'fwd_mean' : ('Fwd Pkt Len Mean','smean'),
  'bwd_mean' : ('Bwd Pkt Len Mean','dmean'),
  'src_load' : ('Flow Byts/s',     'sload'),
  'dst_load' : ('Bwd Pkts/s',      'dload'),
}
SFM_FEATS=list(SFM.keys())
print('9 fitur SFM:', SFM_FEATS)

## 1. Client-UNSW: terapkan SFM ke data UNSW SENDIRI

UNSW train/test = dua berkas rilis resmi (disjoint). Tiap klien olah lokal.

In [ ]:
UNSW_TRAIN='../../unswnb-15/data/UNSW_NB15_training-set.csv'
UNSW_TEST ='../../unswnb-15/data/UNSW_NB15_testing-set.csv'
def unsw_to_sfm(df):
    cols={k:v[1] for k,v in SFM.items()}
    out=pd.DataFrame({k: pd.to_numeric(df[cols[k]], errors='coerce') for k in SFM_FEATS})
    out=out.replace([np.inf,-np.inf], np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    y=df['label'].astype(int).values
    return out.astype(float), y
if os.path.exists(UNSW_TRAIN):
    utr=pd.read_csv(UNSW_TRAIN); ute=pd.read_csv(UNSW_TEST)
    Xu_tr,yu_tr=unsw_to_sfm(utr); Xu_te,yu_te=unsw_to_sfm(ute)
    print('UNSW train', Xu_tr.shape, 'test', Xu_te.shape)
else:
    print('(UNSW CSV tidak ditemukan di path ini - jalankan di SageMaker)')

## 2. Client-CIC: terapkan SFM ke data CIC SENDIRI

CIC dari `cleaned_100.pkl` (praproses Paper 1/2). Stratified 70/30 (seed tetap).

In [ ]:
from sklearn.model_selection import train_test_split
CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'  # sesuaikan path SageMaker
def cic_to_sfm(df):
    cols={k:v[0] for k,v in SFM.items()}
    miss=[cols[k] for k in SFM_FEATS if cols[k] not in df.columns]
    if miss: print('WARN kolom CIC hilang:', miss)
    out=pd.DataFrame({k: pd.to_numeric(df.get(cols[k]), errors='coerce') for k in SFM_FEATS})
    out=out.replace([np.inf,-np.inf],np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float)
if os.path.exists(CIC_PKL):
    d=pickle.load(open(CIC_PKL,'rb'))
    cic_df=pd.DataFrame(d['X'], columns=list(d['feature_names'])) if not isinstance(d.get('X'),pd.DataFrame) else d['X']
    benign=d.get('label_mapping',{}).get('Benign',0)
    y_all=(np.asarray(d['y'])!=benign).astype(int)
    Xc_all=cic_to_sfm(cic_df)
    Xc_tr,Xc_te,yc_tr,yc_te=train_test_split(Xc_all,y_all,test_size=0.3,random_state=SEED,stratify=y_all)
    print('CIC train', Xc_tr.shape, 'test', Xc_te.shape)
else:
    print('(CIC cleaned_100.pkl tidak ditemukan - jalankan di SageMaker; cek nama kolom SFM CIC)')

## 3. Statistik per-klien (ANGKA NYATA) -> tabel dataset Paper 4

In [ ]:
rows=[]
def stat(name, X, y, scheme):
    n=len(y); atk=int(np.sum(y==1)); ben=int(np.sum(y==0))
    rows.append({'client':name,'total':n,'attack':atk,'benign':ben,'scheme':scheme})
try:
    stat('Client-UNSW (train)',Xu_tr,yu_tr,'per-dataset (UNSW release train)')
    stat('Client-UNSW (test)', Xu_te,yu_te,'held-out UNSW test')
except NameError: print('UNSW belum dimuat')
try:
    stat('Client-CIC (train)', Xc_tr,yc_tr,'per-dataset (CIC 70/30 seed)')
    stat('Client-CIC (test)',  Xc_te,yc_te,'held-out CIC test')
except NameError: print('CIC belum dimuat')
dfst=pd.DataFrame(rows)
if len(dfst):
    dfst.to_csv(os.path.join(OUTDIR,'dataset_stats_fed.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfst)
print('=== SEL 3 (statistik per-klien) SELESAI ===')

## 4. Simpan partisi SFM per-klien (lokal + S3)

Tiap partisi = data SATU klien saja (privasi: tak digabung). Upload via boto3
ke `unsw-far/federated/data/{cic,unsw}/` untuk dipakai EC2 klien di Fase 2.

In [ ]:
def save_part(prefix, X, y):
    if X is None: return
    p=os.path.join(OUTDIR, prefix+'.npz')
    np.savez_compressed(p, X=np.asarray(X,float), y=np.asarray(y,int), feats=np.array(SFM_FEATS))
    print('saved', p)
for nm,Xv,yv in [('unsw_train',locals().get('Xu_tr'),locals().get('yu_tr')),
                 ('unsw_test', locals().get('Xu_te'),locals().get('yu_te')),
                 ('cic_train', locals().get('Xc_tr'),locals().get('yc_tr')),
                 ('cic_test',  locals().get('Xc_te'),locals().get('yc_te'))]:
    try: save_part(nm, Xv.values if hasattr(Xv,'values') else Xv, yv)
    except Exception as e: print('skip',nm,':',e)
# Upload ke S3 (boto3; JANGAN aws s3 cp di instance). Jalankan bila di env ber-kredensial.
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in os.listdir(OUTDIR):
        if f.endswith('.npz') or f=='dataset_stats_fed.csv':
            sub='unsw' if f.startswith('unsw') else ('cic' if f.startswith('cic') else 'meta')
            s3.upload_file(os.path.join(OUTDIR,f),S3_BUCKET,f'unsw-far/federated/data/{sub}/{f}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/federated/data/'%S3_BUCKET)
except Exception as e: print('(upload S3 dilewati:',e,')')
print('=== SEL 4 (simpan partisi) SELESAI ===')

## 5. Catatan metodologi (untuk naskah)

- Kedua klien memakai **kamus SFM yang sama (public prior)** tetapi menerapkannya
  pada datanya MASING-MASING -> tak ada pertukaran data mentah (privasi FL terjaga).
- Normalisasi (z-score) dilakukan **per-klien lokal** di Fase 2 (opsi (a) Bagian A.1);
  statistik tak dibagikan. Opsi (b)/(c) sebagai ablasi bila perlu.
- Test set per-klien disimpan terpisah; **evaluasi global** memakai gabungan test
  (held-out) HANYA di notebook evaluasi offline Fase 3 (bukan data latih klien).